In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('RetailedClean.csv', header=None)

# set header names
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_Id', 'Country', 'Source_sheet']

df.head()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.shape
df.info()

C:\Users\rutvik kihla\AppData\Local\Temp\ipykernel_15564\172315959.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('RetailedClean.csv', header=None)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1033036 non-null  object        
 1   StockCode     1033036 non-null  object        
 2   Description   1028761 non-null  object        
 3   Quantity      1033036 non-null  int64         
 4   InvoiceDate   1033036 non-null  datetime64[ns]
 5   Price         1033036 non-null  float64       
 6   Customer_Id   797885 non-null   float64       
 7   Country       1033036 non-null  object        
 8   Source_sheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


In [6]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY
# Recency = customer's last purchased
# Frequency = how much time buy
# Monetary = total money spend / Total Spending

rfm_data = df[(df['Customer_Id'].notna()) & (df['Quantity'] > 0)].copy()
rfm_data

snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_Id').agg(
    recency = ('InvoiceDate', lambda d : (snapshot_date - d.max()).days ),
    frequency = ('Invoice', 'nunique'),
    Monetary = ('Quantity', lambda x : (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

rfm

,Customer_Id,recency,frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40
...,...,...,...,...
5876,18283.0,3,22,2664.90
5877,18284.0,431,1,461.68
5878,18285.0,660,1,427.00
5879,18286.0,476,2,1296.43


In [32]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(rfm['recency'], 5, labels=[1,2,3,4,5]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)
# rfm['R_Score'] , rfm['F_Score'], rfm['M_Score']

# - function for customers seperation
def customer_segment(row):
    # Rule 1: High scores across the board
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champion'
    #   recently buy and frequntly buy   
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    # not buy recently, before cust buy frequently but 'not buy recently'    
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3: 
        return 'At Risk'
    # cust buy recently, but not frequently buy
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2: 
        return 'New'
    # cust recently midium , frequency of buying low
    elif row['R_Score'] == 3 and row['F_Score'] <= 2: 
        return 'Needs Attention'
    # buyers which not active
    else: 
        return 'Lost'

rfm['Segment'] = rfm.apply(customer_segment, axis=1)
# check customer leave or not - cust not buy in more than last 90 days it's means cust leav.
rfm['IsChurned'] = rfm['recency'] > 90

In [46]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[(df['Customer_Id'].notna()) & (df['Quantity'] > 0)]
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] =  df['Quantity'] * df['Price']

# Merge Table
df_final = df.merge(rfm[['Customer_Id', 'recency', 'frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
                    on='Customer_Id', how='left')

# Export to Excel
df_final.to_excel('IkeaDashboardData.xlsx', index=False)
print('Export Successfully.......')

Export Successfully.......
